In [1]:
import pandas as pd
import sqlite3
import torch
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

# 1. Load and Prepare Data
conn = sqlite3.connect('../data/warehouse.db')
df = pd.read_sql_query("SELECT subject, body, tag_1 AS queue FROM tickets", conn)
conn.close()

df = df.dropna(subset=['subject', 'body', 'queue'])
df['text'] = df['subject'] + " " + df['body']

valid_queues = df['queue'].value_counts()[df['queue'].value_counts() > 1].index
df = df[df['queue'].isin(valid_queues)]

# 2. Encode Labels
labels = df['queue'].unique().tolist()
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}
df['label'] = df['queue'].map(label2id)

X_train, X_test, y_train, y_test = train_test_split(
    df['text'].tolist(), df['label'].tolist(), test_size=0.2, random_state=42, stratify=df['label']
)

# 3. Create HuggingFace Datasets
train_dataset = Dataset.from_dict({'text': X_train, 'label': y_train})
test_dataset = Dataset.from_dict({'text': X_test, 'label': y_test})

# 4. Tokenization
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def tokenize_function(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=128)

train_tokenized = train_dataset.map(tokenize_function, batched=True)
test_tokenized = test_dataset.map(tokenize_function, batched=True)

# 5. Load Model
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=len(labels), id2label=id2label, label2id=label2id
)

# 6. Training Arguments (Lightweight for local execution)
training_args = TrainingArguments(
    output_dir='../models/distilbert_results',
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='macro')
    return {"accuracy": acc, "macro_f1": f1}

# 7. Train and Evaluate
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    compute_metrics=compute_metrics,
)

print("Starting DistilBERT fine-tuning...")
trainer.train()

# 8. Save the Winning Model
results = trainer.evaluate()
print("\n--- DISTILBERT RESULTS ---")
print(f"Accuracy: {results['eval_accuracy']:.4f} | Macro-F1: {results['eval_macro_f1']:.4f}")

if results['eval_accuracy'] > 0.6777: # Comparing against your TF-IDF queue baseline
    print("DistilBERT won! Saving model...")
    trainer.save_model('../models/winning_triage_model')
    tokenizer.save_pretrained('../models/winning_triage_model')
else:
    print("TF-IDF remains the winner. We will use TF-IDF for the FastAPI backend.")

d:\TicketsSupportIntelligence\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 18040.79it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those p

Starting DistilBERT fine-tuning...


d:\TicketsSupportIntelligence\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,0.935450,0.774922,0.213609


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.38it/s]
d:\TicketsSupportIntelligence\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
No log,0.935450,1,0.774922,0.213609



--- DISTILBERT RESULTS ---
Accuracy: 0.7749 | Macro-F1: 0.2136
DistilBERT won! Saving model...


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.11s/it]
